In [ ]:
# Практична робота. Microsoft Fabric: pipeline завантаження продажів із ноутбуком і поштовим звітом

StatementMeta(, bb4defc7-39fd-44a8-8ead-0d0edb6bb289, 15, Finished, Available, Finished, False)

In [ ]:
table_name = "dbo.sales"

StatementMeta(, bb4defc7-39fd-44a8-8ead-0d0edb6bb289, 16, Finished, Available, Finished, False)

In [3]:
# Creating schema and reading data

from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DateType, DoubleType
)

sales_schema = StructType([
    StructField("SalesOrderNumber", StringType(), True),
    StructField("SalesOrderLineNumber", IntegerType(), True),
    StructField("OrderDate", DateType(), True),
    StructField("CustomerName", StringType(), True),
    StructField("EmailAddress", StringType(), True),
    StructField("Item", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("UnitPrice", DoubleType(), True),
    StructField("TaxAmount", DoubleType(), True)
])

sales_df = (
    spark.read
    .schema(sales_schema)
    .option("header", "true")
    .option("delimiter", ",")
    .option("dateFormat", "yyyy-MM-dd")
    .csv("Files/new_data/*.csv")
)

display(sales_df.limit(20))

StatementMeta(, bb4defc7-39fd-44a8-8ead-0d0edb6bb289, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f4595765-f1e8-46ea-8d56-69a9b148c0df)

In [4]:
sales_df.printSchema()

StatementMeta(, bb4defc7-39fd-44a8-8ead-0d0edb6bb289, 18, Finished, Available, Finished, False)

root
 |-- SalesOrderNumber: string (nullable = true)
 |-- SalesOrderLineNumber: integer (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- EmailAddress: string (nullable = true)
 |-- Item: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- TaxAmount: double (nullable = true)



In [10]:
# Transform the data

from pyspark.sql.functions import (
    year, month, split, col
)

transformed_df = sales_df.select(
    col("SalesOrderNumber"),
    col("SalesOrderLineNumber"),
    col("OrderDate"),
    year(col("OrderDate")).cast("int").alias("Year"),
    month(col("OrderDate")).cast("int").alias("Month"),
    split(col("CustomerName"), " ")[0].alias("FirstName"),
    split(col("CustomerName"), " ", 2)[1].alias("LastName"),
    col("EmailAddress"),
    col("Item"),
    col("Quantity"),
    col("UnitPrice"),
    col("TaxAmount")
)

display(transformed_df.limit(20))

StatementMeta(, bb4defc7-39fd-44a8-8ead-0d0edb6bb289, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d5b2eca3-19fa-4509-a965-b6f062ca7b0b)

In [ ]:
# Writing data to the table

(
transformed_df.write
.format("delta")
.mode("overwrite")
.option("overwriteSchema", "true")
.saveAsTable(table_name)
)

print(f"Data successfully written to {table_name}")

StatementMeta(, bb4defc7-39fd-44a8-8ead-0d0edb6bb289, 20, Finished, Available, Finished, False)

Data successfully written to dbo.sales
